In [1]:
from deep_translator import GoogleTranslator

translated = GoogleTranslator(source='auto', target='ru').translate('Hello world')
print(translated)

/mnt/d/PycharmProjects/SpamFilter/ubuntu_venv/lib/python3.12/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.2)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


ProxyError: HTTPSConnectionPool(host='translate.google.com', port=443): Max retries exceeded with url: /m?tl=ru&sl=auto&q=Hello+world (Caused by ProxyError('Unable to connect to proxy', NewConnectionError("HTTPSConnection(host='10.255.255.254', port=10809): Failed to establish a new connection: [Errno 111] Connection refused")))

In [ ]:
import csv
import requests
csv.field_size_limit(10 * 1024 * 1024)

with open('combined_data.csv', 'r', encoding='utf-8') as f_in, \
     open('translated_data_big.csv', 'w', encoding='utf-8', newline='') as f_out:

    reader = csv.DictReader(f_in)
    writer = csv.writer(f_out)
    writer.writerow(['label', 'translated'])
    rows = list(reader)
    print(rows[0])
    print('считали')
    rows.sort(key=lambda row: len(row['text']))
    print('сортируем')
    print(rows[0])
    for index, row in enumerate(rows):
        text = row['text']
        if len(text) <= 500:
            continue
        # Truncate to 5000 characters (the max allowed)
        # if len(text) >= 5000:
        #     text = text[:4999]
        response = requests.post("http://127.0.0.1:8008/translate",json={"q": row['text'], "source": "en", "target": "ru"})
        translated = response.json()["translatedText"]
        writer.writerow([row['label'], translated])
        if index%100 == 0:
            print(index)

{'label': '1', 'text': 'ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread flack desperate iambic hadron heft quell yoghurt bunkmate divert afterimage'}
считали
сортируем
{'label': '1', 'text': 'l'}


In [18]:
import requests

response = requests.post(
    "http://127.0.0.1:5000/translate",
    json={"q": "Hello world", "source": "en", "target": "ru"}
)
print(response.json()["translatedText"])

Привет, мир


In [7]:
import csv
import asyncio
import aiohttp

csv.field_size_limit(10 * 1024 * 1024)

INPUT_FILE = "combined_data.csv"
OUTPUT_FILE = "translated_data.csv"
TRANSLATE_URL = "http://127.0.0.1:8008/translate"

MAX_CONCURRENT_REQUESTS = 4
MAX_RETRIES = 3


async def translate_row(session: aiohttp.ClientSession, row: dict, semaphore: asyncio.Semaphore):
    label = row.get("label", "")
    text = row.get("text", "")

    if not text:
        return label, "", "empty_text"

    # Ограничение длины текста
    if len(text) > 4999:
        text = text[:4999]

    payload = {
        "q": text,
        "source": "en",
        "target": "ru"
    }

    last_error = ""

    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                async with session.post(TRANSLATE_URL, json=payload) as resp:
                    raw_body = await resp.text()

                    if resp.status != 200:
                        last_error = f"http_{resp.status}: {raw_body[:300]}"
                        continue

                    # Пытаемся разобрать JSON только после проверки статуса
                    try:
                        data = await resp.json(content_type=None)
                    except Exception as e:
                        last_error = f"bad_json: {e}; body={raw_body[:300]}"
                        continue

                    translated = data.get("translatedText")
                    if translated is None:
                        last_error = f"no_translatedText_in_response: {data}"
                        continue

                    return label, translated, ""

            except asyncio.TimeoutError:
                last_error = "timeout"
            except aiohttp.ClientError as e:
                last_error = f"client_error: {e}"
            except Exception as e:
                last_error = f"unexpected_error: {e}"

            await asyncio.sleep(0.5 * attempt)

    return label, "", last_error


async def main():
    # Чтение входного CSV
    with open(INPUT_FILE, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    timeout = aiohttp.ClientTimeout(total=60, connect=10, sock_read=50)
    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT_REQUESTS)

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f_out:
        writer = csv.writer(f_out)
        writer.writerow(["label", "translated"])

        async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
            tasks = [
                asyncio.create_task(translate_row(session, row, semaphore))
                for row in rows
            ]

            processed = 0
            for task in asyncio.as_completed(tasks):
                label, translated, error = await task
                writer.writerow([label, translated])

                processed += 1
                if processed % 100 == 0:
                    print(f"Processed {processed} rows")

    print("Done.")


# В Jupyter:
await main()

CancelledError: 

In [8]:
import csv
csv.field_size_limit(10 * 1024 * 1024)
original_file = open("combined_data.csv", 'r', encoding='utf-8')
new_file = open("translated_data.csv", 'r', encoding='utf-8')
new_big_file = open("translated_data_big.csv", 'r', encoding='utf-8')
print(len(original_file.readlines()), len(new_file.readlines()))
print(len(new_file.readlines())+len(new_big_file.readlines()), )
original_file = open("combined_data.csv", 'r', encoding='utf-8')
reader = csv.DictReader(original_file)
count = 0
for row in reader:
    if len(row['text']) > 500:
        count+=1
print(count)

778530 88715
60069


In [2]:
import pandas as pd

# Чтение двух CSV файлов
df1 = pd.read_csv('translated_data.csv')
df2 = pd.read_csv('translated_data_big.csv')

# Объединение данных
combined_df = pd.concat([df1, df2], ignore_index=True)

# Сохранение результата в новый CSV файл
combined_df.to_csv('translated_combined_file.csv', index=False)


In [4]:
import pandas as pd

# Чтение двух CSV файлов
df1 = pd.read_csv('combined_data.csv')
df2 = pd.read_csv('translated_combined_file.csv')

# Объединение данных
combined_df = pd.concat([df1, df2], ignore_index=True)

# Сохранение результата в новый CSV файл
combined_df.to_csv('translated_and_english_combined_file.csv', index=False)

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Test")
    .master("local[*]")
    .getOrCreate()
)

print(spark.range(5).collect())
spark.stop()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.